<a href="https://colab.research.google.com/github/shreyasrivatsan/REC-POAI-LAB-EXPERIMENTS/blob/main/category_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time

# The NPC's memory (the "Model")
last_seen_room = None

# A loop that runs the game for 3 turns
for turn in range(1, 4):
    print(f"\n--- TURN {turn} ---")

    # Get input from you (the player)
    action = input("Type 'peek' to let the guard see you, or 'hide': ").lower()

    if action == "peek":
        player_is_visible = True
        current_room = input("Which room are you in? (e.g., Kitchen, Bedroom): ")
    else:
        player_is_visible = False

    # --- NPC DECISION PROCESS ---
    if player_is_visible:
        last_seen_room = current_room  # NPC updates its memory!
        print(f"Guard: 'I see you in the {last_seen_room}! Charging at you!'")

    elif last_seen_room is not None:
        print(f"Guard: 'I lost you... but I am going to search the {last_seen_room}!'")

    else:
        print("Guard: 'Everything is quiet. Just patrolling the castle.'")

    time.sleep(1)  # Pauses for 1 second to make it feel like a real game



--- TURN 1 ---
Type 'peek' to let the guard see you, or 'hide': peek
Which room are you in? (e.g., Kitchen, Bedroom): balcony
Guard: 'I see you in the balcony! Charging at you!'

--- TURN 2 ---
Type 'peek' to let the guard see you, or 'hide': hide
Guard: 'I lost you... but I am going to search the balcony!'

--- TURN 3 ---
Type 'peek' to let the guard see you, or 'hide': peek
Which room are you in? (e.g., Kitchen, Bedroom): kitchen
Guard: 'I see you in the kitchen! Charging at you!'


In [ ]:
# The agent's internal state (the 'model' of the world) stored as global memory
agent_memory = {}
rooms_list = []
current_location = ""


def update_model(location, status):
    """Updates the global internal memory."""
    global current_location
    current_location = location
    agent_memory[location] = status


def choose_action():
    """Decision logic based purely on the global memory state."""
    # Rule 1: Clean if the current room is dirty
    if agent_memory[current_location] == "Dirty":
        return "Suck"

    # Rule 2: Move to any room that is dirty or unknown
    for room in rooms_list:
        if agent_memory[room] in ["Dirty", "Unknown"]:
            return f"Move to {room}"

    # Rule 3: Stop when memory confirms everything is clean
    return "Stop"


# --- Simulation Setup & Execution ---
print("--- Simple Vacuum Setup ---")
room_input = input("Enter rooms separated by commas (e.g., Bedroom, Kitchen): ")
rooms_list = [r.strip() for r in room_input.split(",") if r.strip()]

# Set up the environment and initialize unknown memory
environment = {}
for room in rooms_list:
    status_choice = input(f"Is {room} dirty? (y/n): ").strip().lower()
    environment[room] = "Dirty" if status_choice == "y" else "Clean"
    agent_memory[room] = "Unknown"

current_location = rooms_list[0]

print("\n--- Starting Cleaning Cycle ---")
for step in range(1, 20):
    current_status = environment[current_location]
    print(f"\n[Step {step}] Vacuum is in {current_location}. Sensor: {current_status}")

    # Process state and choose action
    update_model(current_location, current_status)
    action = choose_action()

    print(f"Agent Memory: {agent_memory}")
    print(f"Action: {action}")

    if action == "Suck":
        environment[current_location] = "Clean"
    elif action.startswith("Move to"):
        current_location = action.replace("Move to ", "")
    elif action == "Stop":
        print("\nAll done! Every room is verified clean.")
        break


--- Simple Vacuum Setup ---
Enter rooms separated by commas (e.g., Bedroom, Kitchen): Living, Bedroom1, Bedroom2, Dining, Kitchen, Study, Balcony
Is Living dirty? (y/n): y
Is Bedroom1 dirty? (y/n): n
Is Bedroom2 dirty? (y/n): y
Is Dining dirty? (y/n): y
Is Kitchen dirty? (y/n): y
Is Study dirty? (y/n): n
Is Balcony dirty? (y/n): y

--- Starting Cleaning Cycle ---

[Step 1] Vacuum is in Living. Sensor: Dirty
Agent Memory: {'Living': 'Dirty', 'Bedroom1': 'Unknown', 'Bedroom2': 'Unknown', 'Dining': 'Unknown', 'Kitchen': 'Unknown', 'Study': 'Unknown', 'Balcony': 'Unknown'}
Action: Suck

[Step 2] Vacuum is in Living. Sensor: Clean
Agent Memory: {'Living': 'Clean', 'Bedroom1': 'Unknown', 'Bedroom2': 'Unknown', 'Dining': 'Unknown', 'Kitchen': 'Unknown', 'Study': 'Unknown', 'Balcony': 'Unknown'}
Action: Move to Bedroom1

[Step 3] Vacuum is in Bedroom1. Sensor: Clean
Agent Memory: {'Living': 'Clean', 'Bedroom1': 'Clean', 'Bedroom2': 'Unknown', 'Dining': 'Unknown', 'Kitchen': 'Unknown', 'Study':

In [ ]:
import time

# A 5x5 map: . is a road, # is a wall block
MAP = [
    [".", ".", ".", "#", "."],
    [".", "#", ".", "#", "."],
    [".", "#", ".", ".", "."],
    [".", ".", "#", "#", "."],
    ["#", ".", ".", ".", "."]
]

def draw_map(agent_pos, goal_pos):
    print("\n" * 2)  # Clears space in terminal
    for y in range(5):
        row_str = ""
        for x in range(5):
            if (x, y) == agent_pos:
                row_str += " A "  # Agent
            elif (x, y) == goal_pos:
                row_str += " G "  # Goal
            else:
                row_str += f" {MAP[y][x]} "  # Shows empty road or wall
        print(row_str)
    print("-" * 15)

def get_distance(pos1, pos2):
    # Calculates how many grid squares away the goal is
    return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])

def navigate_to_goal(start, goal):
    current = start
    visited = {current}  # Keeps track of where we have been to avoid loops

    draw_map(current, goal)
    time.sleep(1)

    while current != goal:
        cx, cy = current
        best_move = None
        best_distance = float('inf')

        # Check all 4 possible directions: Up, Down, Left, Right
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nx, ny = cx + dx, cy + dy
            neighbor = (nx, ny)

            # 1. Check if the move stays inside the 5x5 map boundaries
            if 0 <= nx < 5 and 0 <= ny < 5:
                # 2. Check if the space is a clear road and not a wall (#)
                if MAP[ny][nx] != "#" and neighbor not in visited:

                    # 3. See how close this step gets us to the goal
                    dist = get_distance(neighbor, goal)
                    if dist <= best_distance or best_move is None:
                        best_distance = dist
                        best_move = neighbor

        # If the agent gets trapped with no valid moves
        if best_move is None:
            print("Target blocked! No safe route available.")
            return

        # Move the agent to the best chosen neighbor
        current = best_move
        visited.add(current)

        # Animate the movement
        draw_map(current, goal)
        time.sleep(1)  # Pauses for 1 second to create a live animation

    print("Goal reached successfully!")

# --- Execution ---
print("--- Advanced Visual GPS System ---")
print("Map size is 5x5. Choose coordinates from 0 to 4.")
print("Avoid walls (#) like (1,1) or (3,0).")

start_x = int(input("Enter starting X (0-4): "))
start_y = int(input("Enter starting Y (0-4): "))
goal_x = int(input("Enter goal X (0-4): "))
goal_y = int(input("Enter goal Y (0-4): "))

navigate_to_goal((start_x, start_y), (goal_x, goal_y))


--- Advanced Visual GPS System ---
Map size is 5x5. Choose coordinates from 0 to 4.
Avoid walls (#) like (1,1) or (3,0).
Enter starting X (0-4): 4
Enter starting Y (0-4): 0
Enter goal X (0-4): 0
Enter goal Y (0-4): 2



 .  .  .  #  A 
 .  #  .  #  . 
 G  #  .  .  . 
 .  .  #  #  . 
 #  .  .  .  . 
---------------



 .  .  .  #  . 
 .  #  .  #  A 
 G  #  .  .  . 
 .  .  #  #  . 
 #  .  .  .  . 
---------------



 .  .  .  #  . 
 .  #  .  #  . 
 G  #  .  .  A 
 .  .  #  #  . 
 #  .  .  .  . 
---------------



 .  .  .  #  . 
 .  #  .  #  . 
 G  #  .  A  . 
 .  .  #  #  . 
 #  .  .  .  . 
---------------



 .  .  .  #  . 
 .  #  .  #  . 
 G  #  A  .  . 
 .  .  #  #  . 
 #  .  .  .  . 
---------------



 .  .  .  #  . 
 .  #  A  #  . 
 G  #  .  .  . 
 .  .  #  #  . 
 #  .  .  .  . 
---------------



 .  .  A  #  . 
 .  #  .  #  . 
 G  #  .  .  . 
 .  .  #  #  . 
 #  .  .  .  . 
---------------



 .  A  .  #  . 
 .  #  .  #  . 
 G  #  .  .  . 
 .  .  #  #  . 
 #  .  .  .  . 
-------

In [ ]:
# --- Configuration / Preferences ---
TARGET_SPEED = 100  # km/h

def evaluate_utility(action, current_speed, obstacle_distance, weather):
    """Calculates a utility score for a given action using metric units."""
    utility = 0

    # Predict the outcome of the action
    predicted_speed = current_speed
    if action == "ACCELERATE":
        predicted_speed += 15
    elif action == "BRAKE":
        predicted_speed -= 25
    elif action == "EMERGENCY_BRAKE":
        predicted_speed = 0

    # Weather multiplier scales danger penalties
    danger_multiplier = 2.0 if weather == "RAIN" else 1.0

    # 1. SAFETY PREFERENCE (Highest Priority)
    if obstacle_distance < 10 and action != "EMERGENCY_BRAKE":
        utility -= 150 * danger_multiplier
    elif obstacle_distance < 25 and action == "ACCELERATE":
        utility -= 60 * danger_multiplier

    # 2. SPEED PREFERENCE (Efficiency)
    speed_error = abs(TARGET_SPEED - predicted_speed)
    utility -= speed_error * 0.5

    # 3. COMFORT PREFERENCE (Avoid jerky movements)
    if action == "EMERGENCY_BRAKE" and obstacle_distance > 25:
        utility -= 30

    return utility

def choose_action(current_speed, obstacle_distance, weather):
    """Selects the action with the highest utility score."""
    possible_actions = ["MAINTAIN", "ACCELERATE", "BRAKE", "EMERGENCY_BRAKE"]
    best_action = None
    highest_utility = float('-inf')

    for action in possible_actions:
        score = evaluate_utility(action, current_speed, obstacle_distance, weather)
        if score > highest_utility:
            highest_utility = score
            best_action = action

    return best_action

# --- Simulation Control Loop ---
print(f"Interactive Utility Agent Active (Target Speed: {TARGET_SPEED} km/h)\n")

# Initial States (Metric)
weather = "CLEAR"
speed = 60
distance = 50

while True:
    # 1. Run the agent and show current dashboard
    final_decision = choose_action(speed, distance, weather)
    print(f"📊 DASHBOARD: Speed = {speed} km/h | Distance = {distance} m | Weather = {weather}")
    print(f"🤖 AGENT ACTION -> {final_decision}")

    # 2. Apply physical consequences of the agent's action
    if final_decision == "ACCELERATE":
        speed += 15
    elif final_decision == "BRAKE":
        speed = max(0, speed - 25)
    elif final_decision == "EMERGENCY_BRAKE":
        speed = 0

    if speed > 0:
        distance = max(0, distance - int(speed * 0.3))

    # Check simple end-game conditions
    if distance <= 0 and speed > 0:
        print("💥 CRASH! The car hit the obstacle.")
        break
    elif speed == 0 and distance <= 2:
        print("🛑 SUCCESS! The car successfully stopped safely before the obstacle.")
        break

    # 3. Explicit Dynamic User Prompt with Inline Command Guide
    print("\n[COMMAND GUIDE: 'w rain', 'w clear', 's 80' (speed), 'd 45' (distance), 'exit']")
    user_input = input("Press Enter to step, or type a command: ").strip().lower()
    print("-" * 60)

    if user_input == 'exit':
        break
    elif user_input.startswith('w '):
        new_weather = user_input.split(' ')[1].upper()
        if new_weather in ["CLEAR", "RAIN"]:
            weather = new_weather
            print(f"🌤️ Systems updated: Weather changed to {weather}")
        else:
            print(f"❌ Unknown weather choice! Only [clear] or [rain] work.")
    elif user_input.startswith('s '):
        try:
            speed = int(user_input.split(' ')[1])
            print(f"⚡ Driver Intervention: Speed manually set to {speed} km/h")
        except (ValueError, IndexError):
            print("❌ Invalid speed command. Example format: s 80")
    elif user_input.startswith('d '):
        try:
            distance = int(user_input.split(' ')[1])
            print(f"📍 Sensor Update: New obstacle detected at {distance} meters")
        except (ValueError, IndexError):
            print("❌ Invalid distance command. Example format: d 45")



Interactive Utility Agent Active (Target Speed: 100 km/h)

📊 DASHBOARD: Speed = 60 km/h | Distance = 50 m | Weather = CLEAR
🤖 AGENT ACTION -> ACCELERATE

[COMMAND GUIDE: 'w rain', 'w clear', 's 80' (speed), 'd 45' (distance), 'exit']
Press Enter to step, or type a command: s 75
------------------------------------------------------------
⚡ Driver Intervention: Speed manually set to 75 km/h
📊 DASHBOARD: Speed = 75 km/h | Distance = 28 m | Weather = CLEAR
🤖 AGENT ACTION -> ACCELERATE

[COMMAND GUIDE: 'w rain', 'w clear', 's 80' (speed), 'd 45' (distance), 'exit']
Press Enter to step, or type a command: 46
------------------------------------------------------------
📊 DASHBOARD: Speed = 90 km/h | Distance = 1 m | Weather = CLEAR
🤖 AGENT ACTION -> EMERGENCY_BRAKE
🛑 SUCCESS! The car successfully stopped safely before the obstacle.
